# Objective 1: build the 10-K dataset (laptop steps)

Run the cells from top to bottom. Use the **.venv** kernel (top right).

This notebook covers the steps that run on the laptop. Each step reads files and writes files, so you can close the notebook between steps and pick up later.

## Step 1.1: download the 20 filings from SEC EDGAR

- **The 8 training companies:** their FY2024 and FY2025 10-Ks.
- **The 4 unseen companies** (CRWV, FIG, ORCL, AMD): their FY2025 10-K only.
- **The fiscal year** is the calendar year in which the filing's period ends, so NVIDIA's FY2025 is the year ending 2025-01-26. 10-K/A amendments are skipped.
- **Each filing** goes to `data/raw/<TICKER>_FY<year>/`, together with a `metadata.json` file.
- **`data/filings.csv`** lists all 20 filings with their accession numbers (the SEC's ID for each filing), so anyone can re-download the same ones.

In [ ]:
# Setup: find the project folder, load config.yaml, and tell EDGAR who you are.
import csv
import json
import os
import time
from pathlib import Path

import yaml
from edgar import Company, set_identity

# The notebook may run from the notebooks/ folder, so walk up until config.yaml is found.
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
MANIFEST = ROOT / "data" / "filings.csv"

config = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))

identity = os.environ.get("SEC_IDENTITY")
assert identity, "Set the SEC_IDENTITY environment variable, then restart VS Code."
set_identity(identity)  # the SEC requires a name and email with every request

print("Project folder:", ROOT)

In [ ]:
# The list of (ticker, fiscal year) pairs to download.
unseen = config["heldout_companies"] + config["test_only_companies"]
training = [c for c in config["companies"] if c not in unseen]

pairs = [(t, fy) for t in training for fy in config["fiscal_years"]]  # 8 companies x 2 years
pairs += [(t, config["test_fy"]) for t in unseen]                   # 4 companies x FY2025

print(len(pairs), "filings:", pairs)

In [ ]:
# Two functions: find the right 10-K, and save it.

def find_10k(ticker, fiscal_year):
    """The single original 10-K whose period ends in `fiscal_year`."""
    filings = Company(ticker).get_filings(form="10-K", amendments=False)
    matches = [f for f in filings
               if f.form == "10-K" and str(f.period_of_report)[:4] == str(fiscal_year)]
    if len(matches) != 1:
        periods = [str(f.period_of_report) for f in filings]
        raise RuntimeError(f"{ticker} FY{fiscal_year}: expected 1 10-K, found {len(matches)}; periods {periods}")
    return matches[0]


def download(ticker, fiscal_year, force=False):
    """Save the main 10-K document, any Exhibit 13, and metadata.json. Returns the metadata."""
    folder = RAW / f"{ticker}_FY{fiscal_year}"
    meta_path = folder / "metadata.json"
    if meta_path.exists() and not force:
        print(f"{ticker} FY{fiscal_year}: already downloaded")
        return json.loads(meta_path.read_text(encoding="utf-8"))

    filing = find_10k(ticker, fiscal_year)
    folder.mkdir(parents=True, exist_ok=True)

    # The main 10-K document
    main_name = filing.primary_documents[0].document
    (folder / main_name).write_text(filing.html(), encoding="utf-8")

    # Exhibit 13: only if the company put its financial statements there instead of Item 8
    exhibit_13 = [a for a in filing.exhibits if (a.document_type or "").upper().startswith("EX-13")]
    for a in exhibit_13:
        content = a.download()
        if isinstance(content, bytes):
            content = content.decode("utf-8", errors="replace")
        (folder / a.document).write_text(content, encoding="utf-8")

    meta = {
        "ticker": ticker,
        "fiscal_year": fiscal_year,
        "form": filing.form,
        "company": filing.company,
        "cik": filing.cik,
        "period_of_report": str(filing.period_of_report),
        "filing_date": str(filing.filing_date),
        "accession_number": filing.accession_no,
        "url": filing.homepage_url,
        "main_document": main_name,
        "exhibit_13": [a.document for a in exhibit_13],
        "exhibits": [{"type": a.document_type or "", "document": a.document} for a in filing.exhibits],
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"{ticker} FY{fiscal_year}: saved {main_name} (period {meta['period_of_report']}, filed {meta['filing_date']})")
    return meta

In [ ]:
# Download all 20. Takes about a minute.
# FORCE = True downloads again even if a filing is already saved.
FORCE = False

rows, failures = [], []
for ticker, fy in pairs:
    try:
        rows.append(download(ticker, fy, force=FORCE))
    except Exception as error:  # keep going, and report at the end
        failures.append(f"{ticker} FY{fy}: {error}")
        print(f"{ticker} FY{fy}: FAILED: {error}")
    time.sleep(0.5)  # stay well under the SEC's limit of 10 requests per second

print(f"\n{len(rows)} downloaded, {len(failures)} failed")
for f in failures:
    print("  ", f)

In [ ]:
# Write data/filings.csv: the list of filings, which goes into git.
fields = ["ticker", "fiscal_year", "company", "period_of_report", "filing_date",
          "accession_number", "main_document", "exhibit_13", "url"]
with MANIFEST.open("w", newline="", encoding="utf-8") as out:
    writer = csv.DictWriter(out, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    for row in rows:
        writer.writerow({**row, "exhibit_13": ";".join(row["exhibit_13"])})

print("Wrote", MANIFEST.relative_to(ROOT), "with", len(rows), "filings")

In [ ]:
# Check: is each file a complete 10-K? Looks for the main Items and any Exhibit 13.
import html
import re

for row in rows:
    folder = RAW / f"{row['ticker']}_FY{row['fiscal_year']}"
    raw_html = (folder / row["main_document"]).read_text(encoding="utf-8")
    text = html.unescape(re.sub(r"<[^>]+>", " ", raw_html))  # strip the HTML tags
    text = re.sub(r"\s+", " ", text)
    items = [i for i in ["1A", "1C", "7", "7A", "8", "9A"]
             if re.search(r"Item\s*" + i + r"\b", text, re.I)]
    print(f"{folder.name:13} {len(text.split()) / 1000:4.0f}k words   Items found: {', '.join(items):22} Exhibit 13: {row['exhibit_13'] or 'none'}")